In [ ]:
!pip install transformers datasets evaluate torch pandas numpy matplotlib scikit-learn

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
import torch
import pandas as pd
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [ ]:
MODEL_CKPT = os.getenv("MODEL_CKPT", "distilbert/distilbert-base-uncased")
os.makedirs("../model", exist_ok=True)
os.makedirs("../results", exist_ok=True)

label2id = {'Informative': 0, 'Misinformative': 1}
id2label = {0: 'Informative', 1: 'Misinformative'}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted'),
        'precision': precision_score(labels, predictions, average='weighted'),
        'recall': recall_score(labels, predictions, average='weighted')
    }

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True, max_length=512)

with open("../config.json", "r") as f:
    all_configs = json.load(f)

args_keys = [key for key in all_configs.keys() if key.startswith("args")]
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

train_df = pd.read_csv("../results/train.csv")
val_df = pd.read_csv("../results/val.csv")

train_dataset = Dataset.from_pandas(train_df).map(
    preprocess_function, batched=True
).remove_columns(['title']).with_format('torch')

val_dataset = Dataset.from_pandas(val_df).map(
    preprocess_function, batched=True
).remove_columns(['title']).with_format('torch')

In [ ]:
training_results = []

for args_key in args_keys:
    
    config_args = all_configs[args_key]
    
    config = AutoConfig.from_pretrained(MODEL_CKPT, label2id=label2id, id2label=id2label)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, config=config).to(device)
    
    training_args = TrainingArguments(
        output_dir=config_args["output_dir"],
        num_train_epochs=config_args["num_train_epochs"],
        learning_rate=config_args["learning_rate"],
        per_device_train_batch_size=config_args["per_device_train_batch_size"],
        per_device_eval_batch_size=config_args["per_device_eval_batch_size"],
        weight_decay=config_args["weight_decay"],
        disable_tqdm=config_args["disable_tqdm"],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        compute_metrics=compute_metrics, 
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    print(model.config)

    trainer.train()
    
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
results_df = pd.DataFrame(training_results)
results_df.to_csv('../results/model_training_metrics.csv', index=False)
print("\n" + results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics = ['accuracy', 'f1', 'precision', 'recall']
titles = ['Accuracy', 'F1 Score', 'Precision', 'Recall']

for ax, metric, title in zip(axes.flatten(), metrics, titles):
    ax.bar(results_df['config'], results_df[metric])
    ax.set_xlabel('Configuration')
    ax.set_ylabel(title)
    ax.set_title(f'{title} by Configuration')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/metrics_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, col, title in zip(axes.flatten(), 
                          ['learning_rate', 'batch_size', 'weight_decay', 'epochs'],
                          ['Learning Rate', 'Batch Size', 'Weight Decay', 'Epochs']):
    df_sorted = results_df.sort_values(col)
    ax.plot(df_sorted[col], df_sorted['f1'], 'o-', linewidth=2, markersize=8)
    ax.set_xlabel(title)
    ax.set_ylabel('F1 Score')
    ax.set_title(f'F1 Score vs {title}')
    if col == 'learning_rate':
        ax.set_xscale('log')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/hyperparameter_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
best_config = results_df.loc[results_df['f1'].idxmax()]
print("Best Configuration: \n")
print(f"Config: {best_config['config']}")
print(f"LR: {best_config['learning_rate']}, BS: {best_config['batch_size']}, "
      f"Epochs: {best_config['epochs']}, WD: {best_config['weight_decay']}")
print(f"F1: {best_config['f1']:.4f}, Acc: {best_config['accuracy']:.4f}")